In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [2]:
df_games = pd.read_parquet("../data/games.parquet")
df_games

,Event,Site,White,Black,Result,WhiteElo,BlackElo,TimeControl,ECO,Opening,Termination,UTCDate,UTCTime
0,Rated Classical tournament https://lichess.org...,https://lichess.org/2lAzqJ9u,Asia_01,Pakus,0-1,1622,1587,600+0,?,?,Abandoned,2016.12.31,23:00:01
1,Rated Classical tournament https://lichess.org...,https://lichess.org/5utpphVs,cronopiodano,Magnataroniell,1-0,1708,1690,600+0,B10,Caro-Kann Defense: Euwe Attack,Time forfeit,2016.12.31,23:00:01
2,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/xUn53e0k,MarlonTeixeira21,caguertti,0-1,1661,1614,300+0,B30,Sicilian Defense: Old Sicilian,Time forfeit,2016.12.31,23:00:01
3,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/Z5NSdZX4,WinterSnowStorm,theivi,1-0,1968,1893,300+0,D35,"Queen's Gambit Declined: Exchange Variation, P...",Normal,2016.12.31,23:00:01
4,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/RyirpGs3,walt_b,james205,0-1,1204,1461,300+0,C44,Ponziani Opening,Normal,2016.12.31,23:00:01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10680703,Rated Blitz game,https://lichess.org/VnBlWWrf,arruru,michalisem,0-1,1816,1880,300+0,B21,Sicilian Defense: McDonnell Attack,Normal,2017.01.31,22:59:56
10680704,Rated Bullet game,https://lichess.org/gDXG5waf,jozarov,agadmator,0-1,2214,1995,60+0,A31,"English Opening: Symmetrical Variation, Anti-B...",Time forfeit,2017.01.31,22:59:56
10680705,Rated Classical game,https://lichess.org/dSsRJyZe,asquithbom,glopezp25,0-1,1272,1442,600+0,B01,Scandinavian Defense: Main Line,Normal,2017.01.31,22:59:51
10680706,Rated Classical game,https://lichess.org/VAwHk00V,TEBEPIZDA2,CaptainJaco,1-0,1455,1251,600+0,B20,Sicilian Defense: Bowdler Attack,Normal,2017.01.31,22:59:59


In [3]:
df_games.shape

(10680708, 13)

In [4]:
df_games.head()

,Event,Site,White,Black,Result,WhiteElo,BlackElo,TimeControl,ECO,Opening,Termination,UTCDate,UTCTime
0,Rated Classical tournament https://lichess.org...,https://lichess.org/2lAzqJ9u,Asia_01,Pakus,0-1,1622,1587,600+0,?,?,Abandoned,2016.12.31,23:00:01
1,Rated Classical tournament https://lichess.org...,https://lichess.org/5utpphVs,cronopiodano,Magnataroniell,1-0,1708,1690,600+0,B10,Caro-Kann Defense: Euwe Attack,Time forfeit,2016.12.31,23:00:01
2,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/xUn53e0k,MarlonTeixeira21,caguertti,0-1,1661,1614,300+0,B30,Sicilian Defense: Old Sicilian,Time forfeit,2016.12.31,23:00:01
3,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/Z5NSdZX4,WinterSnowStorm,theivi,1-0,1968,1893,300+0,D35,"Queen's Gambit Declined: Exchange Variation, P...",Normal,2016.12.31,23:00:01
4,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/RyirpGs3,walt_b,james205,0-1,1204,1461,300+0,C44,Ponziani Opening,Normal,2016.12.31,23:00:01


In [5]:
df_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 10680708 entries, 0 to 10680707
Data columns (total 13 columns):
 #   Column       Dtype
---  ------       -----
 0   Event        str  
 1   Site         str  
 2   White        str  
 3   Black        str  
 4   Result       str  
 5   WhiteElo     str  
 6   BlackElo     str  
 7   TimeControl  str  
 8   ECO          str  
 9   Opening      str  
 10  Termination  str  
 11  UTCDate      str  
 12  UTCTime      str  
dtypes: str(13)
memory usage: 2.5 GB


In [6]:
df_games.duplicated().sum()

np.int64(0)

Numeric values such as 'WhiteElo' and 'BlackElo' should be converted to int. Also date and time should be converted to datetime. Datetime step will be performed on a clean data.

In [7]:
elo_cols = ["WhiteElo", "BlackElo"]

for c in elo_cols:
    bad = pd.to_numeric(df_games[c], errors="coerce").isna()
    print(c, df_games.loc[bad, c].value_counts(dropna=False).head(10))

WhiteElo Series([], Name: count, dtype: int64)
BlackElo Series([], Name: count, dtype: int64)


In [8]:
df_games[elo_cols] = (
    df_games[elo_cols]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int16")
)

In [9]:
df_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 10680708 entries, 0 to 10680707
Data columns (total 13 columns):
 #   Column       Dtype
---  ------       -----
 0   Event        str  
 1   Site         str  
 2   White        str  
 3   Black        str  
 4   Result       str  
 5   WhiteElo     Int16
 6   BlackElo     Int16
 7   TimeControl  str  
 8   ECO          str  
 9   Opening      str  
 10  Termination  str  
 11  UTCDate      str  
 12  UTCTime      str  
dtypes: Int16(2), str(11)
memory usage: 2.3 GB


In [10]:
df_games.describe(include="all")

,Event,Site,White,Black,Result,WhiteElo,BlackElo,TimeControl,ECO,Opening,Termination,UTCDate,UTCTime
count,10680708,10680708,10680708,10680708,10680708,10680708.0,10680708.0,10680708,10680708,10680708,10680708,10680708,10680708
unique,6324,10680708,212579,204854,4,<NA>,<NA>,887,495,2993,5,32,86400
top,Rated Blitz game,https://lichess.org/2lAzqJ9u,Beserking101,Beserking101,1-0,<NA>,<NA>,300+0,A00,Van't Kruijs Opening,Normal,2017.01.08,22:00:02
freq,4003140,1,3418,3413,5294709,<NA>,<NA>,1976667,709843,219765,7293796,387327,1278
mean,NaN,NaN,NaN,NaN,NaN,1663.941055,1662.757682,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,276.254426,277.088381,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,706.0,715.0,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,1478.0,1474.0,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,1664.0,1663.0,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,1850.0,1850.0,NaN,NaN,NaN,NaN,NaN,NaN


There are 32 unique dates.

In [11]:
ts = df_games["UTCDate"] + " " + df_games["UTCTime"]
print("first:", ts.min())
print("last: ", ts.max())

dec31 = df_games.loc[df_games["UTCDate"] == "2016.12.31", "UTCTime"]
print(dec31.str[:2].value_counts())

first: 2016.12.31 23:00:00
last:  2017.01.31 22:59:59
UTCTime
23    10907
Name: count, dtype: int64


The file starts at 2016-12-31 23:00 UTC and ends at 2017-01-31 22:59 UTC, and all 10,907 games dated 2016-12-31 were played between 23:00 and 23:59 UTC. The file therefore covers exactly one month shifted by one hour, which is consistent with month boundaries defined in UTC+1.
These games are kept. Train/test splits will use datetime cut-offs, not day numbers.

In [12]:
df_games.nunique()

Event              6324
Site           10680708
White            212579
Black            204854
Result                4
WhiteElo           2189
BlackElo           2202
TimeControl         887
ECO                 495
Opening            2993
Termination           5
UTCDate              32
UTCTime           86400
dtype: int64

In [13]:
df_games.isnull().sum()

Event          0
Site           0
White          0
Black          0
Result         0
WhiteElo       0
BlackElo       0
TimeControl    0
ECO            0
Opening        0
Termination    0
UTCDate        0
UTCTime        0
dtype: int64

In [14]:
text_cols = df_games.select_dtypes(include=["object", "string"]).columns

for c in text_cols:
    s = df_games[c]
    odd = s[~s.str.contains(r"[A-Za-z0-9]", regex=True, na=False)]
    if len(odd):
        print(c, odd.value_counts().to_dict())

White {'____________': 80}
Black {'____________': 81}
Result {'*': 1558}
TimeControl {'-': 51326}
ECO {'?': 34964}
Opening {'?': 34964}


In [15]:
name = "____________"
u = df_games[(df_games["White"] == name) | (df_games["Black"] == name)].copy()
u["elo"] = u["WhiteElo"].where(u["White"] == name, u["BlackElo"])

print(u["elo"].describe())
print(u["TimeControl"].value_counts().head())
u.sort_values(["UTCDate", "UTCTime"])[["UTCDate", "UTCTime", "elo", "TimeControl"]].head(20)

count          161.0
mean     1847.024845
std        32.477868
min           1773.0
25%           1829.0
50%           1850.0
75%           1868.0
max           1914.0
Name: elo, dtype: Float64
TimeControl
360+8     64
360+9     63
600+0     31
300+10     3
Name: count, dtype: int64


,UTCDate,UTCTime,elo,TimeControl
26624,2017.01.01,01:41:54,1842,360+8
249589,2017.01.01,20:24:14,1850,360+8
277861,2017.01.01,22:00:03,1840,360+8
282878,2017.01.01,22:18:23,1845,360+8
284220,2017.01.01,22:23:17,1854,360+8
288128,2017.01.01,22:38:23,1842,360+8
993973,2017.01.04,00:01:19,1841,360+8
1025392,2017.01.04,03:08:57,1829,360+8
1027607,2017.01.04,03:25:02,1822,360+8
1031987,2017.01.04,03:59:33,1837,360+8


"____________" is a valid username, not a placeholder.

Missing values are encoded as "?" or "-" instead of nulls. They are converted to NA so that standard missing-value tools detect them.

In [16]:
SENTINELS = ["?", "-"]
text_cols = df_games.select_dtypes(include=["object", "string"]).columns

df_count = len(df_games)
counts = df_games[text_cols].isin(SENTINELS).sum()
print(counts[counts > 0])

df_games[text_cols] = df_games[text_cols].replace(SENTINELS, pd.NA)

TimeControl    51326
ECO            34964
Opening        34964
dtype: int64


In [17]:
missing = (
    df_games.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing", "mean": "share"})
    .astype({"missing": int})
    .query("missing > 0")
    .sort_values("missing", ascending=False)
)
missing["share"] = (missing["share"] * 100).round(3)
missing

,missing,share
TimeControl,51326,0.481
ECO,34964,0.327
Opening,34964,0.327


In [18]:
df_games["Result"].value_counts()

Result
1-0        5294709
0-1        4954027
1/2-1/2     430414
*             1558
Name: count, dtype: int64

In [19]:
pd.crosstab(df_games["Result"]=="*", df_games["Termination"])

Termination,Abandoned,Normal,Rules infraction,Time forfeit,Unterminated
Result,,,,,
False,52453,7293796,1353,3331548,0
True,1508,0,0,0,50


In [20]:
df_games["Termination"].value_counts()

Termination
Normal              7293796
Time forfeit        3331548
Abandoned             53961
Rules infraction       1353
Unterminated             50
Name: count, dtype: int64

In [21]:
pd.crosstab(df_games["ECO"].isna(), df_games["Termination"])

Termination,Abandoned,Normal,Rules infraction,Time forfeit,Unterminated
ECO,,,,,
False,24900,7288990,1353,3330451,50
True,29061,4806,0,1097,0


In [22]:
df_games.loc[df_games["TimeControl"].isna(), "Event"].value_counts().head()

Event
Rated Correspondence game    51326
Name: count, dtype: int64

Missing values are not dropped. Missing TimeControl occurs only in correspondence games, which are removed by a separate rule. Missing ECO/Opening are kept. These are games that ended before an opening could be classified. They will get an "unknown" category at the feature stage.

In [23]:
df_games["event_type"] = (
    df_games["Event"].str.replace(r"\s*https?://\S+", "", regex=True).str.strip()
)
df_games["event_type"].value_counts()

event_type
Rated Blitz game              4003140
Rated Classical game          2619965
Rated Bullet game             1980420
Rated Blitz tournament         921691
Rated Bullet tournament        788297
Rated Classical tournament     315869
Rated Correspondence game       51326
Name: count, dtype: int64

Decisions:
- Games without a proper ending are removed: no result (*), Unterminated, Abandoned and Rules infraction. Their outcome does not come from play.
- Correspondence games are removed. With days per move, time pressure plays no role, so this is a different game. All games with missing TimeControl are correspondence games.

In [24]:
rules = {
    "no_result": df_games["Result"] == "*",
    "abandoned": df_games["Termination"] == "Abandoned",
    "rules_infraction": df_games["Termination"] == "Rules infraction",
    "unterminated": df_games["Termination"] == "Unterminated",
    "correspondence": df_games["event_type"] == "Rated Correspondence game",
}

summary = pd.Series({k: int(m.sum()) for k, m in rules.items()}, name="rows_flagged")
remove = pd.concat(rules, axis=1).any(axis=1)

print(summary)
print(f"Total removed: {remove.sum():,} ({remove.mean():.2%})")

df_clean = df_games[~remove].copy().reset_index(drop=True)

no_result            1558
abandoned           53961
rules_infraction     1353
unterminated           50
correspondence      51326
Name: rows_flagged, dtype: int64
Total removed: 106,459 (1.00%)


In [25]:
df_clean["utc_dt"] = pd.to_datetime(
    df_clean["UTCDate"] + " " + df_clean["UTCTime"],
    format="%Y.%m.%d %H:%M:%S",
    errors="coerce",
)
print(df_clean["utc_dt"].isna().sum())

df_clean = df_clean.drop(columns=["UTCDate", "UTCTime"])

0


In [26]:
df_clean.describe(include="all")

,Event,Site,White,Black,Result,WhiteElo,BlackElo,TimeControl,ECO,Opening,Termination,event_type,utc_dt
count,10574249,10574249,10574249,10574249,10574249,10574249.0,10574249.0,10574249,10569026,10569026,10574249,10574249,10574249
unique,6263,10574249,206810,199462,3,<NA>,<NA>,886,494,2989,2,6,NaN
top,Rated Blitz game,https://lichess.org/5utpphVs,Beserking101,Beserking101,1-0,<NA>,<NA>,300+0,A00,Van't Kruijs Opening,Normal,Rated Blitz game,NaN
freq,4002410,1,3381,3365,5246073,<NA>,<NA>,1965072,703778,217883,7251893,4002410,NaN
mean,NaN,NaN,NaN,NaN,NaN,1664.119465,1662.888163,NaN,NaN,NaN,NaN,NaN,2017-01-16 16:20:02.152624
min,NaN,NaN,NaN,NaN,NaN,706.0,715.0,NaN,NaN,NaN,NaN,NaN,2016-12-31 23:00:00
25%,NaN,NaN,NaN,NaN,NaN,1477.0,1474.0,NaN,NaN,NaN,NaN,NaN,2017-01-08 20:04:05
50%,NaN,NaN,NaN,NaN,NaN,1664.0,1663.0,NaN,NaN,NaN,NaN,NaN,2017-01-16 17:46:29
75%,NaN,NaN,NaN,NaN,NaN,1850.0,1850.0,NaN,NaN,NaN,NaN,NaN,2017-01-24 12:22:13
max,NaN,NaN,NaN,NaN,NaN,3119.0,3117.0,NaN,NaN,NaN,NaN,NaN,2017-01-31 22:59:59


In [27]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10574249 entries, 0 to 10574248
Data columns (total 13 columns):
 #   Column       Dtype         
---  ------       -----         
 0   Event        str           
 1   Site         str           
 2   White        str           
 3   Black        str           
 4   Result       str           
 5   WhiteElo     Int16         
 6   BlackElo     Int16         
 7   TimeControl  str           
 8   ECO          str           
 9   Opening      str           
 10  Termination  str           
 11  event_type   str           
 12  utc_dt       datetime64[us]
dtypes: Int16(2), datetime64[us](1), str(10)
memory usage: 2.3 GB
